# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. The exploration uses unique `@id` values for all dataset entities as recommended by Croissant best practices.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata using the Dataset API
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate the record sets (typically tables, data files, etc.) and their fields.
All entity references are made using their distinct `@id` as per Croissant conventions.

In [ ]:
# List all record sets and their fields/columns using @id

print("Record sets in the dataset:")
recordsets = dataset.record_sets
for rs in recordsets:
    print(f"\nRecordSet name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '')}")
    print("  Fields (@id):")
    for field in rs.fields:
        col_id = getattr(field, 'id', None)
        print(f"    - {col_id} (name: {getattr(field, 'name', '')}, type: {getattr(field, 'data_type', '')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll load all record sets and store each as a DataFrame.

In [ ]:
# Get all record set @id values
record_set_ids = [rs.id for rs in recordsets]
dataframes = {}

for rs in recordsets:
    print(f"Loading: {rs.name} (@id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"  Columns: {df.columns.tolist()}")
    print(f"  First 2 records:\n{df.head(2).to_string(index=False)}\n")
# For demonstration below, pick the first record set
primary_record_set_id = record_set_ids[0] if record_set_ids else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps. We'll pick a numeric field (for example, the total GAD-7 score column if present), demonstrate filtering, normalization, and grouping operations.

Remember to reference columns by their Croissant `@id` value. Adjust the field selection as relevant to this dataset.

In [ ]:
import numpy as np

# Let's select relevant variable IDs for the EDA. We'll infer possible candidates by looking for GAD-7, PHQ-9, or PSQ-related columns.

df = dataframes[primary_record_set_id]
print(f"Columns in primary record set (@id={primary_record_set_id}):\n", df.columns)

# Let's try to pick a numeric field automatically, e.g., GAD-7, PHQ-9 or a score field
gad7_candidates = [col for col in df.columns if ('gad7' in col.lower() or 'gad_7' in col.lower()) and not col.endswith('id')]
phq9_candidates = [col for col in df.columns if ('phq9' in col.lower() or 'phq_9' in col.lower()) and not col.endswith('id')]
score_candidates = [col for col in df.columns if ('score' in col.lower() or 'sum' in col.lower() or 'total' in col.lower()) and not col.endswith('id')]
possible_numeric = gad7_candidates or phq9_candidates or score_candidates

if len(possible_numeric) > 0:
    numeric_field_id = possible_numeric[0]  # Use the first candidate field
    print(f"Using numeric field: {numeric_field_id} (Croissant @id)")
else:
    print("Did not automatically find a GAD-7/PHQ-9 score column. Please set numeric_field_id manually.")
    numeric_field_id = df.columns[0] # fallback

threshold = df[numeric_field_id].mean() if numeric_field_id in df.columns else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a demographic field if present, e.g., gender or age_group
group_fields = [col for col in df.columns if ('gender' in col.lower() or 'sex' in col.lower() or 'age_group' in col.lower())]
if group_fields:
    group_field = group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No grouping field (e.g., gender, sex, age_group) found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot a histogram of the main score field, and if demographic columns are available, a grouped boxplot as well.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id} (Croissant @id)")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by demographic group if available
if group_fields:
    plt.figure(figsize=(6,4))
    sns.boxplot(data=df, x=group_field, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading and exploring a Croissant AI-Ready dataset on mental health survey data from Kilifi County, Kenya. We programmatically referenced all dataset assets by their Croissant `@id`, loaded data into DataFrames, performed basic EDA, and visualized high-level trends.

This approach can be adapted to other Croissant-based datasets for systematic, transparent, and reproducible data analysis.
